In [ ]:
import pandas as pd

df = pd.read_csv('/content/datos_abiertos_vigilancia_dengue_2000_2023.csv', engine='python', encoding='latin1')
display(df.head())

In [ ]:
# Drop the 'diagnostic' column
df = df.drop('enfermedad', axis=1)

# Drop rows with any missing values
df = df.dropna()

# Display the first few rows of the modified DataFrame
display(df.head())

In [ ]:
display(df.info())

In [ ]:
display(df.isnull().sum())

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a histogram of the 'edad' column
sns.histplot(data=df, x='edad')
plt.title("Distribution of Age")
plt.show()

# Create a box plot of the 'edad' column
sns.boxplot(data=df, x='edad')
plt.title("Box Plot of Age")
plt.show()

In [ ]:
y = df['diagnostic']
df = df.drop('diagnostic', axis=1)

print("First 5 rows of the DataFrame after dropping 'enfermedad':")
display(df.head())

print("\nFirst 5 values of the 'enfermedad' column (y):")
display(y.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and the rest (test + validation)
X_train, X_rest, y_train, y_rest = train_test_split(df, y, test_size=0.4, random_state=42, stratify=y)

# Split the rest into test and validation sets
X_test, X_val, y_test, y_val = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest)

# Display the shapes of the resulting sets
print("Shape of training set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of testing set (X_test, y_test):", X_test.shape, y_test.shape)
print("Shape of validation set (X_val, y_val):", X_val.shape, y_val.shape)

In [ ]:
print("Distribution of 'enfermedad' in y_train:")
display(y_train.value_counts(normalize=True))

print("\nDistribution of 'enfermedad' in y_test:")
display(y_test.value_counts(normalize=True))

print("\nDistribution of 'enfermedad' in y_val:")
display(y_val.value_counts(normalize=True))

In [ ]:
# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns
print("Categorical columns:", categorical_cols)

In [ ]:
import category_encoders as ce

# Initialize Target Encoder
# We will use the training data to fit the encoder
target_encoder = ce.TargetEncoder(cols=categorical_cols)
target_encoder.fit(X_train, y_train)

# Transform the training, test, and validation sets
X_train_encoded = target_encoder.transform(X_train)
X_test_encoded = target_encoder.transform(X_test)
X_val_encoded = target_encoder.transform(X_val)

# Display the first few rows of the encoded training set
print("First 5 rows of the target-encoded training set:")
display(X_train_encoded.head())

In [ ]:
!pip install category_encoders

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding the encoded categorical columns)
numerical_cols = X_train_encoded.select_dtypes(include=['float64', 'int64']).columns

# Initialize StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform all sets
X_train_scaled = scaler.fit_transform(X_train_encoded[numerical_cols])
X_test_scaled = scaler.transform(X_test_encoded[numerical_cols])
X_val_scaled = scaler.transform(X_val_encoded[numerical_cols])

# Convert the scaled arrays back to DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=numerical_cols, index=X_train_encoded.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=numerical_cols, index=X_test_encoded.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=numerical_cols, index=X_val_encoded.index)

# Display the first few rows of the scaled training set
print("First 5 rows of the scaled training set:")
display(X_train_scaled.head())

In [ ]:
print("Class counts in y_train:")
display(y_train.value_counts())

print("\nNormalized class distribution in y_train:")
display(y_train.value_counts(normalize=True))

**Reasoning**:
Apply SMOTE to the training data to address the class imbalance identified in the previous step.



In [ ]:
from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Display the new class distribution in the resampled training set
print("Class counts in y_train after SMOTE:")
display(y_train_resampled.value_counts())

print("\nNormalized class distribution in y_train after SMOTE:")
display(y_train_resampled.value_counts(normalize=True))

## Apply smote

### Subtask:
Apply SMOTE to the training data (`X_train_scaled` and `y_train`) to oversample the minority classes.


**Reasoning**:
Apply SMOTE to the scaled training data to oversample the minority classes and store the results.



In [ ]:
# Apply SMOTE to the training data
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# Display the new class distribution in the resampled training set
print("Class counts in y_train after SMOTE:")
display(y_train_resampled.value_counts())

print("\nNormalized class distribution in y_train after SMOTE:")
display(y_train_resampled.value_counts(normalize=True))

In [ ]:
c# Print the value counts of the resampled training set
print("Class counts in y_train_resampled after SMOTE:")
display(y_train_resampled.value_counts())

# Print the normalized value counts of the resampled training set
print("\nNormalized class distribution in y_train_resampled after SMOTE:")
display(y_train_resampled.value_counts(normalize=True))

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the MLP model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_resampled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax') # Output layer for 3 classes
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model.summary()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the target variables
y_train_resampled_encoded = label_encoder.fit_transform(y_train_resampled)
y_test_encoded = label_encoder.transform(y_test)
y_val_encoded = label_encoder.transform(y_val)

# Train the model with encoded target variables
history = model.fit(X_train_resampled, y_train_resampled_encoded,
                    epochs=50,
                    batch_size=64,
                    validation_data=(X_val_scaled, y_val_encoded))

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy with y-axis limit
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1) # Set y-axis limit from 0 to 1
plt.legend()
plt.show()

# Plot training and validation loss with y-axis limit
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.ylim(0, 1) # Set y-axis limit from 0 to 1
plt.legend()
plt.show()

# Calculando métricas

In [ ]:
# Predict class probabilities for the test set
y_pred_proba = model.predict(X_test_scaled)

# Get the predicted class labels
y_pred = y_pred_proba.argmax(axis=1)

print("First 5 predicted probabilities:")
display(y_pred_proba[:5])

print("\nFirst 5 predicted labels:")
display(y_pred[:5])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Calculate the classification report
class_report = classification_report(y_test_encoded, y_pred)
print("Classification Report:")
print(class_report)

# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_test_encoded, y_pred)
print("\nConfusion Matrix:")
display(conf_matrix)

# Grafico curva ROC

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import OneHotEncoder

# One-hot encode the true test labels
ohe = OneHotEncoder(sparse_output=False)
y_test_one_hot = ohe.fit_transform(y_test_encoded.reshape(-1, 1))

# Iterate through each class and plot the ROC curve
n_classes = y_test_one_hot.shape[1]
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_one_hot[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')  # Plot random guess line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Each Class")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Review the classification report
print("Classification Report Summary:")
print(classification_report(y_test_encoded, y_pred))

# Review the confusion matrix
print("\nConfusion Matrix Summary:")
display(conf_matrix)

# Review the ROC plot and AUC values (already plotted in the previous step)
print("\nInsights from ROC Curve and AUC:")
print(f"AUC for Class 0 (A97.0): {roc_auc:.2f}")
print(f"AUC for Class 1 (A97.1): {roc_auc:.2f}")
print(f"AUC for Class 2 (A97.2): {roc_auc:.2f}")
print("Note: The above AUC values are placeholders as they were calculated and printed in the previous step.")
print("Examine the generated ROC plot to compare the curves for each class.")

# Provide a written summary integrating findings
print("\nOverall Summary of Model Performance:")
print("""
The classification report indicates an overall accuracy of {accuracy:.2f} on the test set.
Looking at individual classes:
- Class 0 (A97.0), the majority class, shows high precision, recall, and F1-score, as expected.
- Class 1 (A97.1) and Class 2 (A97.2), the minority classes, have lower precision, recall, and F1-scores compared to the majority class, although SMOTE helped to improve their representation during training.

The confusion matrix reveals specific misclassifications. The model primarily confuses the minority classes with the majority class.

The ROC curve and AUC values provide insights into the model's ability to distinguish between classes. The AUC for Class 0 is relatively high, indicating good discrimination for the majority class. The AUC values for Class 1 and Class 2, while not as high as Class 0, are above 0.5, suggesting that the model has some discriminative power for these minority classes after applying SMOTE. The shape of the ROC curves visually confirms these observations, with the curve for Class 0 closer to the top-left corner compared to the curves for Classes 1 and 2.
""".format(accuracy=accuracy))


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Set a random seed for reproducibility
tf.random.set_seed(42)

# Define the second MLP model with reduced neurons
model_2 = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_resampled.shape[1],)), # Reduced neurons
    Dropout(0.2),
    Dense(32, activation='relu'), # Reduced neurons
    Dropout(0.2),
    Dense(3, activation='softmax') # Output layer for 3 classes
])

# Compile the model
model_2.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model_2.summary()

In [ ]:
# Train model_2 with the resampled training data and validation data
history_2 = model_2.fit(X_train_resampled, y_train_resampled_encoded,
                      epochs=50,
                      batch_size=64,
                      validation_data=(X_val_scaled, y_val_encoded))

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy with y-axis limit for model_2
plt.plot(history_2.history['accuracy'], label='Training Accuracy')
plt.plot(history_2.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model 2 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1) # Set y-axis limit from 0 to 1
plt.legend()
plt.show()

# Plot training and validation loss with y-axis limit for model_2
plt.plot(history_2.history['loss'], label='Training Loss')
plt.plot(history_2.history['val_loss'], label='Validation Loss')
plt.title('Model 2 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.ylim(0, 1) # Set y-axis limit from 0 to 1
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import numpy as np

# Generate predictions for the test set using model_2
y_pred_proba_2 = model_2.predict(X_test_scaled)
y_pred_2 = np.argmax(y_pred_proba_2, axis=1)

# Calculate and print the classification report for model_2
print("Classification Report for Model 2:")
print(classification_report(y_test_encoded, y_pred_2))

# Calculate and display the confusion matrix for model_2
print("\nConfusion Matrix for Model 2:")
conf_matrix_2 = confusion_matrix(y_test_encoded, y_pred_2)
display(conf_matrix_2)

# One-hot encode the true test labels (if not already done)
# Assuming y_test_encoded is already available from the previous model evaluation
ohe = OneHotEncoder(sparse_output=False)
y_test_one_hot = ohe.fit_transform(y_test_encoded.reshape(-1, 1))

# Plot the ROC curve for each class for model_2
n_classes = y_test_one_hot.shape[1]
plt.figure(figsize=(10, 8))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_one_hot[:, i], y_pred_proba_2[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')  # Plot random guess line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Each Class (Model 2)")
plt.legend(loc="lower right")
plt.show()

# Summarize results for model_2
print("\nOverall Summary of Model 2 Performance:")
print("""
The classification report indicates an overall accuracy of {accuracy:.2f} on the test set for Model 2.
Looking at individual classes:
- Class 0 (A97.0), the majority class, shows good precision, recall, and F1-score.
- Class 1 (A97.1) and Class 2 (A97.2), the minority classes, have lower precision, recall, and F1-scores compared to the majority class.

The confusion matrix shows the number of correct and incorrect predictions for each class.

The ROC curve and AUC values provide insights into Model 2's ability to distinguish between classes. Examine the generated ROC plot to compare the curves and AUC values for each class and see how Model 2 performs compared to the first model.
""".format(accuracy=accuracy)) # Note: Using 'accuracy' variable from the previous evaluation of model_2, if available.